## AIS ETA Deviation Analysis
In this notebook we'll analyze ETA discrepancies across Cargo, Tanker and Passenger vessels using AIS observations processed by the project's ETL pipeline and stored in a PostgreSQL database.  
  
The analysis estimates ETA with a simple model using Haversine Distance and Speed Over Ground(SOG). The calculated ETA is then compared against each observation's reported ETA.  
  
The primary objectives are to:
- Quantify the difference between reported and calculated ETAs
- Compare discrepancy patterns across different vessel types  
  
Note: The calculated ETA assumes straight-line travel at near constant speeds. Actual vessel routes and verified arrival timestamps are not modeled/included therefore these discrepancies should not be mistaken as direct measures of arrival prediction accuracy.

In [ ]:
import os
import pandas as pd
import numpy as np
import psycopg
from dotenv import load_dotenv
import seaborn as sns
import matplotlib.pyplot as plt

load_dotenv()

with psycopg.connect(dbname=os.getenv('DB_NAME'), user=os.getenv('DB_USER'), 
   password=os.getenv('DB_PASSWORD'), host=os.getenv('DB_HOST'), port=os.getenv('DB_PORT')) as conn:

    imported = pd.read_sql_query("""
       SELECT ship_type, timestamp, mmsi, dest_lat,
       dest_lon, latitude, longitude, sog, eta
       FROM vessel_eta
       WHERE class IS NOT NULL AND
       dest_lat IS NOT NULL AND
       dest_lon IS NOT NULL AND
       sog > 0;
    """, con=conn)

print(imported.duplicated().sum())

Our queried dataframe contains a substantial amount of duplicates. Since identical observations would produce identical distance estimates and ETA deviations, retaining them would wrongfully weight repeated records in the analysis.

In [ ]:
imported = imported.drop_duplicates().reset_index()
print(imported.shape[0])
print(imported.duplicated().sum())

## Distance And ETA Estimation
Our next step is crucial for the analysis. We'll be implementing a haversine function which calculates the great-circle distance between each vessel's reported coordinates and its mapped destination coordinates using the haversine formula.

The calculation uses the destination coordinates produced by our ETL pipeline. Earth's radius is approximated as 6,371 km and the resulting distances are expressed in kilometers.

Note: Haversine distance represents the shortest path over a spherical Earth. It does not account for coastlines, shipping lanes or other route constraints. Therefore, it provides a simplified distance baseline rather than an estimate of the actual maritime route length.

In [ ]:
pd.set_option('display.float_format', lambda x: f"{x:.2f}")

def haversine_distance(ves_lat: float, ves_lon: float, dest_lat: float, dest_lon: float) -> float:

    R_km = 6371

    lat1 = ves_lat * (np.pi/180)
    lat2 = dest_lat * (np.pi/180)
    lon1 = ves_lon * (np.pi/180)
    lon2 = dest_lon * (np.pi/180)

    d_lat = lat2 - lat1
    d_lon = lon2 - lon1

    a = (np.sin(d_lat/2))**2 + np.cos(lat1) * np.cos(lat2) * (np.sin(d_lon/2))**2
    c = 2 * np.arcsin(np.sqrt(a))
    d = R_km * c

    return d

imported['distance_km'] = haversine_distance(imported['latitude'], imported['longitude'], imported['dest_lat'], imported['dest_lon'])
print(imported['distance_km'].describe())
outliers = imported['distance_km'].sort_values(ascending=False).head(40000)
outliers_df = imported[imported.index.isin(outliers.index)]
print(outliers_df['mmsi'].unique())
print(imported[imported['mmsi'] == '538005945'])

Our haversine function produced some genuinely large distances inside our dataframe which are completely expected. One example is a cargo vessel embarking on a voyage from Samso in Denmark and heading to Vitoria in Brazil. Our next step is evaluating ETAs based on our model. Sidenote: keep in mind SOG is expressed in knots so we'll need to convert it to km/h to match our haversine distance unit(1kn = 1.852km/h).

In [ ]:
imported['travel_time_hours'] = imported['distance_km'] / (imported['sog'] * 1.852)
print(imported['travel_time_hours'].describe())
eta_outliers = imported['travel_time_hours'].sort_values(ascending=False).head(4000)
eta_outliers_df = imported[imported.index.isin(eta_outliers.index)]
print(eta_outliers_df['travel_time_hours'].describe())
print(eta_outliers_df['mmsi'].unique())
print(imported['sog'].describe())
print(eta_outliers_df['sog'].describe())
print(imported['sog'][imported['mmsi'] == '218800000'].describe())

Our current model has one caveat. Although most the time its estimates are entirely reasonable e.x. 18.49h, 40.46h, some of them arent exactly plausible **cough cough 32.137.86h. The reason we have produced such estimates is because vessels do not maintain stable speeds all the time. Take the vessel from the last print() statement for example. Its reported SOG seems completely normal. But if you look closely there are some slowdowns like 0.1kn. When our model is faced with these low speeds it produces enormous travel times. NOAA's AIS processing guidance uses 0.5 kn as the beginning of purposeful motion. Instead of simply filtering out speeds below a certain threshold a more sensible fix would be to calculate an hourly median SOG in order to smooth out noisy SOG values and keep out insignificant speeds below 0.5kn.

In [ ]:
hourly = imported.groupby(['mmsi', 'dest_lat', 'dest_lon']).resample('1h', on='timestamp')['sog'].median().reset_index()
hourly = hourly[hourly['sog'] >= 0.5].reset_index(drop=True)
print(hourly['sog'].describe())
print(hourly.shape[0])
print(hourly['mmsi'].nunique())
print(hourly['sog'].isna().sum())

In order to pick an observation to assign each hourly sog to we'll pick the observation of each vessel closest to the midpoint of that hour.

In [ ]:
imported['hour'] = imported['timestamp'].dt.floor('h')
hourly = hourly.rename(columns={
    'timestamp' : 'hour',
    'sog' : 'hourly_sog'
})
imported_hourly = imported.merge(hourly, how='inner', on=['mmsi', 'hour', 'dest_lat', 'dest_lon'])
print(imported_hourly.shape)
imported_hourly['midpoint_diff'] = (imported_hourly['timestamp'] - (imported_hourly['hour'] + pd.Timedelta(minutes=30))).abs()
idx = imported_hourly.groupby(['mmsi', 'hour'])['midpoint_diff'].idxmin()
hourly_representative = imported_hourly.loc[idx].reset_index(drop=True)
hourly_representative['estimated_hours'] = hourly_representative['distance_km'] / (hourly_representative['hourly_sog'] * 1.852)
print(hourly_representative)
print(hourly_representative['estimated_hours'].describe())

Our revised model is working a lot better. We can see that remaining hours like 32k has gone down to 1830 hours which is still 76 days but we cannot guarantee 100% accuracy. Another example is vessel 636092742. Its original estimated time was 60.72h while its revised estimated time is 8.99h. Our next step would be to measure the discrepancy between the reported ETA and the revised estimate.

In [ ]:
hourly_representative['reported_hours'] = (hourly_representative['eta'] - hourly_representative['timestamp']).dt.total_seconds() / 3600
hourly_representative['eta_discrepancy'] = hourly_representative['estimated_hours'] - hourly_representative['reported_hours']
print(hourly_representative['eta_discrepancy'].describe())
worst300 = hourly_representative.sort_values('eta_discrepancy', ascending=True).head(300)
print(worst300)
print(hourly_representative['reported_hours'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

The discrepancy distribution is pretty conflicted. Inspecting the 300 smallest discrepancies (counting negatives) we can see that these observations are not exactly accurate. A vessel cannot have an ETA of over 1 year let alone be this close to port during that time. These observations suggest malformed ETAs or false crew input. However some of them could genuinely be long-distance voyages. In order to avoid ruling these out we'll use a distance + eta filter. Id say an eta of over 1 month combined with a distance of less than 1000km is a pretty reasonable filter.

In [ ]:
suspect_eta = hourly_representative[(hourly_representative['distance_km'] < 1000) & (hourly_representative['reported_hours'] > 720)]
print(suspect_eta.shape[0])
print(worst300.index.isin(suspect_eta.index).sum())
hourly_representative = hourly_representative[~hourly_representative.index.isin(suspect_eta.index)]
print(hourly_representative['eta_discrepancy'].describe())

The filtering has meaningfully improved our discrepancy distribution. However some negative extreme values remain. Lets investigate them before acting on anything else

In [ ]:
worst60 = hourly_representative.sort_values('eta_discrepancy', ascending=True).head(60)
print(worst60['mmsi'].nunique())
print(worst60)

5 vessels represent the 60 smallest discrepancies. However 2 of them caught my eye. MMSI:354290000 is traveling at 11 knots with a remaining distance of around 5.7k km and a reported ETA of 213 days. A 7 month voyage is highly unlikely. The second one MMSI:219004903 is even more suspicious having a reported ETA of 29 days while being less than 10km outside of its reported destination. The other 3 vessels are more than likely long distance voyages based on their observations and behavior. It would be pretty reasonable to remove suspicious observations associated with the two highly suspicious vessels from our dataframe.

In [ ]:
hourly_representative = hourly_representative[(hourly_representative['mmsi'] != '354290000') & (hourly_representative['mmsi'] != '219004903')]
print(hourly_representative['eta_discrepancy'].describe())
print(hourly_representative.sort_values('eta_discrepancy', ascending=False).head(5))

After removing 29 observations associated with the previously mentioned vessels our distribution seems perfectly plausible. The higher discrepancy values still remain. Our next step should be to investigate these values and the vessels behind them.

In [ ]:
print(hourly_representative[hourly_representative['mmsi'] == '538003632'])

These values reveal a genuine limitation to our model. Our model uses an hourly median SOG calculated using speeds from the same hour which protects against individual low speed readings but not against an entire hour of slow movement. Once SOG rises again our model seems to be relatively close to the reported ETA. One could argue about approaching ETA estimation using a model which uses a speed representative of the vessels broader movement throughout the day. We have insufficient observation coverage to attempt such thing ,besides it would add uneccesary complexity to our analysis. Our current ETA discrepancy numbers are calculated at an observation level meaning a very chatty vessel influences the distribution more than a vessel with a handful of observations. Thats why we need to reduce it to vessel level by calculating the median ETA discrepancy of each vessel. Since a vessel can have more than one destination during the observation window we'll first need to reduce the dataframe to vessel + destination level in order to give each destination equal weight and then reduce the already reduced dataframe to vessel level. Having done that every vessel contributes a single number. 

In [ ]:
per_vessel = hourly_representative.groupby(['mmsi', 'dest_lat', 'dest_lon'])['eta_discrepancy'].median().reset_index()
final_discrepancy = per_vessel.groupby('mmsi')['eta_discrepancy'].median().reset_index()
vessel_map = hourly_representative.drop_duplicates('mmsi').set_index('mmsi')['ship_type']
final_discrepancy['ship_type'] = final_discrepancy['mmsi'].map(vessel_map)
print(final_discrepancy.describe())
print(final_discrepancy['ship_type'].value_counts())
cargo_discrepancy = final_discrepancy[final_discrepancy['ship_type'] == 'Cargo']
tanker_discrepancy = final_discrepancy[final_discrepancy['ship_type'] == 'Tanker']
passenger_discrepancy = final_discrepancy[final_discrepancy['ship_type'] == 'Passenger']
print(cargo_discrepancy.describe())
print(tanker_discrepancy.describe())
print(passenger_discrepancy.describe())

sns.scatterplot(
    data = hourly_representative,
    x = 'reported_hours',
    y = 'estimated_hours',
    hue = 'ship_type',
    alpha = 0.65,
    s = 25
)

plt.plot([0, 750], [0, 750], color='red', linestyle='--', label='Perfect agreement')

plt.xlabel('AIS Reported ETA (hours)')
plt.ylabel('Model Estimated ETA (hours)')
plt.title('Estimated vs Reported ETA')
plt.savefig('../figures/eta_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Final Conclusion
In this analysis we identified differences in ETA discrepancies across three different vessel categories. Tankers displayed the largest negative median discrepancy followed by cargo vessels while passenger vessels seemed to agree more frequently with our model while also having substantially lower variability between discrepancies. These findings suggest that the relationship between AIS-reported ETAs and model-based estimates differs across vessel types. From a logical standpoint passenger vessels tend to operate shorter, more regular routes, while cargo vessels and tankers may experience more variation in speed and voyage conditions. However this analysis does not establish ETA accuracy since we dont have any confirmed arrival times.